In [1]:
import os

In [2]:
%pwd

'/workspaces/MLOps-Laptop-Prediction-System/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/workspaces/MLOps-Laptop-Prediction-System'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_dir: Path
    train_data_path: str
    test_data_path: str
    preprocessor_obj_file_path: str

In [6]:
from Laptop_Price_Prediction_System.constants import *
from Laptop_Price_Prediction_System.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath=Config_Filepath, parmas_filepath=Params_Filepath, schema_filepath=Schema_Filepath):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(parmas_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            data_dir = config.data_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            preprocessor_obj_file_path=config.preprocessor_obj_file_path
        )

        return data_transformation_config

In [8]:
import os
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from Laptop_Price_Prediction_System import logger
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

In [ ]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
    
    def fetch_processor(self, text):
        text = text.lower()

        if 'intel core i7' in text:
            return 'Intel Core i7'
        elif 'intel core i5' in text:
            return 'Intel Core i5'
        elif 'intel core i3' in text:
            return 'Intel Core i3'
        elif 'intel' in text:
            return 'Other Intel Processor'
        elif 'amd' in text:
            return 'AMD Processor'
        else:
            return 'Other Processor'
    
    def fetch_memory(self, text):
        ssd = 0
        hdd = 0
        hybrid = 0
        flash = 0

        if pd.isna(text):
            return pd.Series([ssd, hdd, hybrid, flash])

        text = text.lower()
        parts = text.split('+')

        for part in parts:
            part = part.strip()
            num_match = re.search(r'\d+', part)
            value = int(num_match.group())

            if 'gb ssd' in part:
                ssd += value
            elif 'tb hdd' in part:
                hdd += value
            elif 'hybrid' in part:
                hybrid += value
            elif 'flash' in part:
                flash += value

        return pd.Series([ssd, hdd, hybrid, flash])
    
    def categorize_os(self, input_str):
        input_str = str(input_str)

        if 'Windows 7' in input_str or 'Windows 10' in input_str or 'Windows 10 S' in input_str:
            return 'Windows'
        elif 'macOS' in input_str or 'Mac OS X' in input_str:
            return 'Mac'
        elif 'Linux' in input_str:
            return 'Linux'
        elif 'Chrome OS' in input_str:
            return 'Chrome OS'
        else:
            return 'No OS'
    
    def transform(self):
        logger.info("Loading Dataset")
        data = pd.read_csv(self.config.data_dir)

        logger.info("Creating Screen features")
        data['Touchscreen'] = data['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)

        data['IPS'] = data['ScreenResolution'].apply(lambda x: 1 if 'IPS' in x else 0)

        new = data['ScreenResolution'].str.split('x', n=1, expand=True)
        data['X_res'] = new[0].str.extract(r'(\d+)').astype('int')
        data['Y_res'] = new[1].astype('int')

        data['ppi'] = ((data['X_res']**2 + data['Y_res']**2)**0.5) / data['Inches']

        logger.info("Creating CPU features")
        data['Cpu Name'] = data['Cpu'].apply(lambda x: x.split()[0])

        data['Cpu Speed (GHz)'] = data['Cpu'].str.extract(r'(\d+\.\d+)GHz').astype(float)
        data['Cpu Speed (GHz)'] = data['Cpu Speed (GHz)'].fillna(data['Cpu Speed (GHz)'].mean())

        data['Cpu brand'] = data['Cpu'].apply(self.fetch_processor)

        logger.info("Transforming RAM")
        data.rename(columns={'Ram': 'Ram (GB)'}, inplace=True)
        data['Ram (GB)'] = data['Ram (GB)'].str.replace('GB', '').astype(int)

        logger.info("Transforming Memory")
        data[['SSD', 'TB HDD', 'TB Hybrid', 'Flash Storage']] = data['Memory'].apply(self.fetch_memory)

        logger.info("Creating GPU feature")
        data['Gpu Brand'] = data['Gpu'].apply(lambda x: x.split()[0])

        data['OS'] = data['OpSys'].apply(self.categorize_os)

        data['Weight'] = data['Weight'].str.replace('kg', '').astype(float)

        logger.info("Dropping unused columns")
        data.drop(columns=['ScreenResolution', 'Cpu', 'Memory', 'Gpu', 'OpSys', 'X_res', 'Y_res'], inplace=True)

        X = data.drop(columns=["Price"])
        y = data["Price"]

        logger.info("Splitting Dataset")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        train_df = X_train.copy()
        train_df['Price'] = y_train

        test_df = X_test.copy()
        test_df['Price'] = y_test

        train_df.to_csv(self.config.train_data_path, index = False)
        test_df.to_csv(self.config.test_data_path, index = False)
    
    def create_preprocessor(self):
        logger.info("Starting Data Preprocessing")
        train_df = pd.read_csv(self.config.train_data_path)

        X = train_df.drop(columns='Price')

        categorical_columns = X.select_dtypes(include=['object']).columns
        numerical_columns = X.select_dtypes(include=['number']).columns

        preprocessor = ColumnTransformer(transformers=[
            ('cat', OneHotEncoder(sparse_output=False, drop='first'), categorical_columns),
            ('num', StandardScaler(), numerical_columns)
        ])
        
        logger.info("Completed Data Preprocessing")

        X_processed = preprocessor.fit(X)
        
        logger.info("Saved Preprocessor Object")
        joblib.dump(X_processed, self.config.preprocessor_obj_file_path)

In [10]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.transform()
    data_transformation.create_preprocessor()
except Exception as e:
    raise e

[2026-07-10 04:43:50,867 : INFO : common : Loaded file: config/config.yaml]
[2026-07-10 04:43:50,880 : INFO : common : Loaded file: params.yaml]
[2026-07-10 04:43:50,883 : INFO : common : Loaded file: schema.yaml]
[2026-07-10 04:43:50,884 : INFO : common : Created directory: artifacts]
[2026-07-10 04:43:50,899 : INFO : common : Created directory: artifacts/data_transformation]
[2026-07-10 04:43:50,900 : INFO : 2105545180 : Loading Dataset]
[2026-07-10 04:43:50,939 : INFO : 2105545180 : Creating Screen features]
[2026-07-10 04:43:50,956 : INFO : 2105545180 : Creating CPU features]
[2026-07-10 04:43:50,968 : INFO : 2105545180 : Transforming RAM]
[2026-07-10 04:43:50,971 : INFO : 2105545180 : Transforming Memory]
[2026-07-10 04:43:51,123 : INFO : 2105545180 : Creating GPU feature]
[2026-07-10 04:43:51,126 : INFO : 2105545180 : Dropping unused columns]
[2026-07-10 04:43:51,129 : INFO : 2105545180 : Splitting Dataset]


/tmp/ipykernel_49107/2105545180.py:123: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(include=['object']).columns
